# Continuous Random Variables

A **continuous** random variable can take any value in an interval. Instead of a PMF, it has a **Probability Density Function** (PDF).

**Key difference from discrete:**
- Discrete: $P(X = x)$ can be positive
- Continuous: $P(X = x) = 0$ for any exact value! Probabilities only exist for **ranges**.

| Concept | Discrete | Continuous |
|---------|----------|------------|
| Likelihood function | PMF: $P(X = x)$ | PDF: $f(x)$ |
| Probability of value | $P(X = x) \geq 0$ | $P(X = x) = 0$ |
| Probability of range | $\sum_{a}^{b} P(X = x)$ | $\int_a^b f(x)\,dx$ |
| Total probability | $\sum P(X = x) = 1$ | $\int_{-\infty}^{\infty} f(x)\,dx = 1$ |
| Expectation | $\sum x \cdot P(X=x)$ | $\int x \cdot f(x)\,dx$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, integrate

print("Libraries loaded!")

---
## 1. From Histogram to PDF

As we increase the number of bins / decrease bin width, a histogram approaches a smooth PDF curve.

In [ ]:
# Transition from histogram to PDF
np.random.seed(42)
data = np.random.normal(loc=5, scale=2, size=10000)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
x = np.linspace(-2, 12, 300)
pdf = stats.norm.pdf(x, loc=5, scale=2)

for ax, n_bins, title in zip(axes, [5, 15, 50, 200],
                              ['5 bins', '15 bins', '50 bins', '200 bins']):
    ax.hist(data, bins=n_bins, density=True, color='#3498db', edgecolor='white',
            alpha=0.6, linewidth=0.5)
    ax.plot(x, pdf, 'r-', linewidth=2.5, label='True PDF')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('x', fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('More Bins → Histogram Approaches the PDF', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. PDF Rules

A valid PDF $f(x)$ must satisfy:
1. $f(x) \geq 0$ for all $x$
2. $\int_{-\infty}^{\infty} f(x)\,dx = 1$

**Important:** $f(x)$ is NOT a probability! It's a density. $f(x)$ CAN be greater than 1.

In [ ]:
# Example: f(x) > 1 is okay
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform(0, 0.5) — PDF = 2 everywhere in support
ax = axes[0]
rv = stats.uniform(loc=0, scale=0.5)
x = np.linspace(-0.5, 1, 300)
ax.plot(x, rv.pdf(x), 'b-', linewidth=2.5)
ax.fill_between(x, rv.pdf(x), alpha=0.3, color='blue')
ax.axhline(1, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Uniform(0, 0.5): f(x) = 2 > 1!', fontsize=12, fontweight='bold')
ax.annotate('f(x) = 2 (this is OK!)', xy=(0.25, 2), fontsize=11,
            ha='center', color='red', fontweight='bold')

# Beta(5, 1) — PDF > 1 near x=1
ax = axes[1]
rv = stats.beta(5, 1)
x = np.linspace(0, 1, 300)
ax.plot(x, rv.pdf(x), 'r-', linewidth=2.5)
ax.fill_between(x, rv.pdf(x), alpha=0.3, color='red')
ax.axhline(1, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Beta(5, 1): PDF exceeds 1 near x=1', fontsize=12, fontweight='bold')
ax.annotate(f'Peak: f(1) = {rv.pdf(0.999):.1f}', xy=(0.95, rv.pdf(0.999)),
            fontsize=11, color='red', fontweight='bold', ha='right')

plt.tight_layout()
plt.show()

print("Remember: Area under the curve = 1 (even if heights > 1)")
for name, rv_check in [("Uniform(0,0.5)", stats.uniform(0, 0.5)), ("Beta(5,1)", stats.beta(5, 1))]:
    area, _ = integrate.quad(rv_check.pdf, -10, 10)
    print(f"  ∫ f(x) dx for {name} = {area:.6f} ✓")

---
## 3. CDF — Cumulative Distribution Function

$$F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t)\,dt$$

The CDF gives the probability that $X$ is at most $x$. Works for both discrete and continuous!

**Probability of a range:**
$$P(a < X \leq b) = F(b) - F(a)$$

In [ ]:
# PDF and CDF side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

rv = stats.norm(loc=0, scale=1)
x = np.linspace(-4, 4, 300)

# Top left: PDF
ax = axes[0, 0]
ax.plot(x, rv.pdf(x), 'b-', linewidth=2.5)
ax.fill_between(x, rv.pdf(x), alpha=0.2, color='blue')
ax.set_title('PDF: f(x)', fontsize=13, fontweight='bold')
ax.set_ylabel('Density', fontsize=12)

# Top right: CDF
ax = axes[0, 1]
ax.plot(x, rv.cdf(x), 'r-', linewidth=2.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.axhline(1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('CDF: F(x) = P(X ≤ x)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative Probability', fontsize=12)

# Bottom left: P(a < X < b) as area
ax = axes[1, 0]
a, b = -1, 1.5
ax.plot(x, rv.pdf(x), 'b-', linewidth=2)
mask = (x >= a) & (x <= b)
ax.fill_between(x[mask], rv.pdf(x[mask]), alpha=0.5, color='orange',
               label=f'P({a} < X < {b}) = {rv.cdf(b) - rv.cdf(a):.4f}')
ax.axvline(a, color='green', linestyle='--', linewidth=1.5)
ax.axvline(b, color='green', linestyle='--', linewidth=1.5)
ax.set_title(f'P({a} < X < {b}) = F({b}) - F({a})', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylabel('Density', fontsize=12)

# Bottom right: CDF difference
ax = axes[1, 1]
ax.plot(x, rv.cdf(x), 'r-', linewidth=2)
ax.plot([b, b], [0, rv.cdf(b)], 'g--', linewidth=1.5)
ax.plot([a, a], [0, rv.cdf(a)], 'g--', linewidth=1.5)
ax.annotate('', xy=(-3.5, rv.cdf(b)), xytext=(-3.5, rv.cdf(a)),
            arrowprops=dict(arrowstyle='<->', color='orange', lw=3))
ax.text(-3.8, (rv.cdf(a) + rv.cdf(b))/2, f'{rv.cdf(b) - rv.cdf(a):.4f}',
        fontsize=11, color='orange', fontweight='bold')
ax.set_title('F(b) - F(a) on the CDF', fontsize=12, fontweight='bold')
ax.set_ylabel('CDF', fontsize=12)

for ax in axes.flat:
    ax.set_xlabel('x', fontsize=11)

plt.tight_layout()
plt.show()

---
## 4. Expectation and Variance (Continuous)

$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x)\,dx$$
$$\text{Var}(X) = \int_{-\infty}^{\infty} (x - \mu)^2 f(x)\,dx = E[X^2] - (E[X])^2$$

In [ ]:
# Compute expectation and variance numerically
from scipy import integrate

# Custom PDF: triangular on [0, 2]
def triangular_pdf(x):
    if 0 <= x <= 1:
        return x
    elif 1 < x <= 2:
        return 2 - x
    return 0.0

# Verify it integrates to 1
area, _ = integrate.quad(triangular_pdf, -10, 10)
print(f"∫ f(x) dx = {area:.6f} ✓")

# E[X]
E_X, _ = integrate.quad(lambda x: x * triangular_pdf(x), -10, 10)
print(f"E[X] = {E_X:.6f}")

# E[X²]
E_X2, _ = integrate.quad(lambda x: x**2 * triangular_pdf(x), -10, 10)
Var_X = E_X2 - E_X**2
print(f"E[X²] = {E_X2:.6f}")
print(f"Var(X) = E[X²] - E[X]² = {Var_X:.6f}")

# Verify with scipy triangular
rv = stats.triang(c=0.5, loc=0, scale=2)
print(f"\nscipy.stats.triang: E[X] = {rv.mean():.6f}, Var(X) = {rv.var():.6f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
x = np.linspace(-0.5, 2.5, 300)
pdf = np.array([triangular_pdf(xi) for xi in x])
ax.plot(x, pdf, 'b-', linewidth=2.5)
ax.fill_between(x, pdf, alpha=0.3, color='blue')
ax.axvline(E_X, color='red', linewidth=2.5, linestyle='--', label=f'E[X] = {E_X:.2f}')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Triangular PDF on [0, 2]', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | Formula |
|---------|---------|
| PDF | $f(x) \geq 0$, $\int f(x)\,dx = 1$ |
| Probability | $P(a < X \leq b) = \int_a^b f(x)\,dx = F(b) - F(a)$ |
| CDF | $F(x) = P(X \leq x) = \int_{-\infty}^x f(t)\,dt$ |
| $P(X = x)$ | Always 0 for continuous |
| Expectation | $E[X] = \int x \cdot f(x)\,dx$ |
| Variance | $\text{Var}(X) = \int (x - \mu)^2 f(x)\,dx$ |

**Next:** Uniform Distribution — the simplest continuous distribution.